# Privacy-Preserving Record Linkage (PPRL) Demonstration

This notebook demonstrates the **Open Link Token PySpark bridge** end to end. It uses the bridge for both tokenization and overlap analysis, while generating the demo exchange config directly in Python so the walkthrough stays inside the notebook.

The walkthrough covers:

1. generating synthetic hospital and pharmacy datasets,
2. building a demo exchange config in Python,
3. tokenizing both datasets with `OpenLinkTokenProcessor.from_exchange_config(...)`,
4. running strict and relaxed overlap analysis with `OpenLinkTokenOverlapAnalyzer.from_exchange_config(...)`, and
5. inspecting one linked record pair.

> **Important demo disclaimer:** This example uses synthetic data and demo-only secrets. Do not reuse this setup in production.

In [ ]:
import json
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

import pandas as pd
from IPython.display import display


def _find_demo_dir() -> Path:
    """Find the demo directory from the current working directory or its parents."""
    start = Path.cwd().resolve()
    candidates = [start, *start.parents]
    for base in candidates:
        if (base / "scripts" / "generate_superhero_datasets.py").exists():
            return base
        demo = base / "demos" / "pprl-superhero-example"
        if (demo / "scripts" / "generate_superhero_datasets.py").exists():
            return demo
    raise FileNotFoundError("Could not locate demos/pprl-superhero-example")


demo_dir = _find_demo_dir()
scripts_dir = demo_dir / "scripts"
datasets_dir = demo_dir / "datasets"
outputs_dir = demo_dir / "outputs"
bridge_outputs_dir = outputs_dir / "pyspark"
bridge_outputs_dir.mkdir(parents=True, exist_ok=True)

try:
    from pyspark.sql import SparkSession
    from pyspark.sql.functions import lit

    from openlinktoken.ec_key_utils import generate_key_pair
    from openlinktoken.exchange_jwe import build_exchange_envelope
    from openlinktoken_pyspark import OpenLinkTokenProcessor
    from openlinktoken_pyspark.overlap_analyzer import OpenLinkTokenOverlapAnalyzer
except Exception as exc:
    raise RuntimeError(
        "This notebook is specifically for the PySpark bridge demo and requires pyspark plus openlinktoken_pyspark."
    ) from exc


def run_command(command: list[str], cwd: Path = demo_dir) -> str:
    """Run a command, print its output, and raise on failure."""
    print("$", " ".join(command))
    result = subprocess.run(
        command,
        cwd=str(cwd),
        capture_output=True,
        text=True,
        check=False,
    )
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        if result.stderr:
            print(result.stderr)
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {' '.join(command)}")
    return result.stdout


def write_single_csv(dataframe, output_path: Path) -> None:
    """Write a Spark DataFrame to one CSV file with a header."""
    temp_dir = output_path.with_suffix("")
    if temp_dir.exists():
        shutil.rmtree(temp_dir)
    dataframe.coalesce(1).write.mode("overwrite").option("header", "true").csv(str(temp_dir))
    csv_file = next(temp_dir.glob("*.csv"))
    shutil.copy2(csv_file, output_path)
    shutil.rmtree(temp_dir)


spark = (
    SparkSession.builder.appName("PPRL-Superhero-Demo")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f"Demo directory:          {demo_dir}")
print(f"Bridge outputs directory: {bridge_outputs_dir}")

## 1. Generate the synthetic datasets

The demo creates:

- `hospital_superhero_data.csv` with 100 records,
- `pharmacy_superhero_data.csv` with 120 records, and
- a known overlap of 40 records.

In [ ]:
run_command([sys.executable, str(scripts_dir / "generate_superhero_datasets.py")])

hospital_df = pd.read_csv(datasets_dir / "hospital_superhero_data.csv")
pharmacy_df = pd.read_csv(datasets_dir / "pharmacy_superhero_data.csv")

print(f"Hospital dataset rows: {len(hospital_df)}")
print(f"Pharmacy dataset rows: {len(pharmacy_df)}")
display(hospital_df.head())
display(pharmacy_df.head())

assert len(hospital_df) == 100
assert len(pharmacy_df) == 120

## 2. Build the demo exchange config in Python

The notebook creates a demo exchange config and private key directly in Python, then reuses them through the bridge factory APIs.

In [ ]:
exchange_config_path = bridge_outputs_dir / "superhero-demo-notebook.exchange.json"
private_key_path = bridge_outputs_dir / "superhero-demo-notebook.private.pem"

sender_private_pem, sender_public_pem = generate_key_pair("P-256")
_, recipient_public_pem = generate_key_pair("P-256")

exchange_config = build_exchange_envelope(
    exchange_name="superhero-demo-notebook",
    hashing_secret=b"SuperHeroHashingKey2024",
    sender_public_pem=sender_public_pem,
    recipient_public_pem=recipient_public_pem,
    curve="P-256",
    created_at=datetime.now(timezone.utc).replace(microsecond=0).isoformat().replace("+00:00", "Z"),
    exchange_id=str(uuid4()),
)

exchange_config_path.write_text(json.dumps(exchange_config, indent=2), encoding="utf-8")
private_key_path.write_bytes(sender_private_pem)

print(f"Exchange config: {exchange_config_path}")
print(f"Private key:     {private_key_path}")
assert exchange_config_path.exists()
assert private_key_path.exists()

## 3. Tokenize both datasets with the PySpark bridge

This uses `OpenLinkTokenProcessor.from_exchange_config(...)`, which resolves the hashing secret and derived transport key from the exchange config.

In [ ]:
processor = OpenLinkTokenProcessor.from_exchange_config(
    exchange_config_path=exchange_config_path,
    private_key_path=private_key_path,
    ring_id="superhero-demo-ring",
)

hospital_spark_df = spark.read.csv(str(datasets_dir / "hospital_superhero_data.csv"), header=True, inferSchema=True)
pharmacy_spark_df = spark.read.csv(str(datasets_dir / "pharmacy_superhero_data.csv"), header=True, inferSchema=True)

hospital_tokens_spark = processor.process_dataframe(hospital_spark_df)
pharmacy_tokens_spark = processor.process_dataframe(pharmacy_spark_df)

hospital_token_rows = hospital_tokens_spark.count()
pharmacy_token_rows = pharmacy_tokens_spark.count()
print(f"Hospital token rows: {hospital_token_rows}")
print(f"Pharmacy token rows: {pharmacy_token_rows}")

hospital_tokens_csv = bridge_outputs_dir / "hospital_tokens.csv"
pharmacy_tokens_csv = bridge_outputs_dir / "pharmacy_tokens.csv"
write_single_csv(hospital_tokens_spark, hospital_tokens_csv)
write_single_csv(pharmacy_tokens_spark, pharmacy_tokens_csv)

hospital_tokens_preview = pd.read_csv(hospital_tokens_csv)
pharmacy_tokens_preview = pd.read_csv(pharmacy_tokens_csv)
display(hospital_tokens_preview.head(10))
display(pharmacy_tokens_preview.head(10))

assert hospital_token_rows == 500
assert pharmacy_token_rows == 600
assert hospital_tokens_preview["RecordId"].nunique() == 100
assert pharmacy_tokens_preview["RecordId"].nunique() == 120
assert hospital_tokens_preview["Token"].str.startswith("olt.V1.").all()
assert pharmacy_tokens_preview["Token"].str.startswith("olt.V1.").all()

## 4. Run strict overlap analysis with the PySpark bridge

This uses `OpenLinkTokenOverlapAnalyzer.from_exchange_config(...)` and requires all five standard tokens (`T1`-`T5`) to match.

In [ ]:
analyzer = OpenLinkTokenOverlapAnalyzer.from_exchange_config(
    exchange_config_path=exchange_config_path,
    private_key_path=private_key_path,
)

strict_rules = ["T1", "T2", "T3", "T4", "T5"]
strict_results = analyzer.analyze_overlap(
    hospital_tokens_spark,
    pharmacy_tokens_spark,
    matching_rules=strict_rules,
    dataset1_name="Hospital",
    dataset2_name="Pharmacy",
)

print(f"Hospital records: {strict_results['total_records_dataset1']}")
print(f"Pharmacy records: {strict_results['total_records_dataset2']}")
print(f"Matching hospital records: {strict_results['matching_records_dataset1']}")
print(f"Matching pharmacy records: {strict_results['matching_records_dataset2']}")
print(f"Overlap percentage: {strict_results['overlap_percentage']:.1f}%")

strict_matches_spark = (
    strict_results["matches"]
    .withColumnRenamed("Hospital_RecordId", "HospitalRecordId")
    .withColumnRenamed("Pharmacy_RecordId", "PharmacyRecordId")
    .withColumn("MatchingTokens", lit("|".join(strict_rules)))
    .withColumn("TokenCount", lit(len(strict_rules)))
)
strict_matches_csv = bridge_outputs_dir / "matching_records.csv"
write_single_csv(strict_matches_spark, strict_matches_csv)

strict_matches_df = pd.read_csv(strict_matches_csv)
display(strict_matches_df.head(10))

assert strict_results["matching_records_dataset1"] == 40
assert strict_results["matching_records_dataset2"] == 40
assert len(strict_matches_df) == 40
assert strict_matches_df["TokenCount"].eq(5).all()

## 5. Run a relaxed bridge match policy

This second bridge example excludes `T4`, so the policy becomes `T1`, `T2`, `T3`, and `T5` must all match.

In [ ]:
relaxed_rules = ["T1", "T2", "T3", "T5"]
relaxed_results = analyzer.analyze_overlap(
    hospital_tokens_spark,
    pharmacy_tokens_spark,
    matching_rules=relaxed_rules,
    dataset1_name="Hospital",
    dataset2_name="Pharmacy",
)

print(f"Strict hospital matches:  {strict_results['matching_records_dataset1']}")
print(f"Relaxed hospital matches: {relaxed_results['matching_records_dataset1']}")

relaxed_matches_spark = (
    relaxed_results["matches"]
    .withColumnRenamed("Hospital_RecordId", "HospitalRecordId")
    .withColumnRenamed("Pharmacy_RecordId", "PharmacyRecordId")
    .withColumn("MatchingTokens", lit("|".join(relaxed_rules)))
    .withColumn("TokenCount", lit(len(relaxed_rules)))
)
relaxed_matches_csv = bridge_outputs_dir / "matching_records_alt.csv"
write_single_csv(relaxed_matches_spark, relaxed_matches_csv)

relaxed_matches_df = pd.read_csv(relaxed_matches_csv)
display(relaxed_matches_df.head(10))

assert relaxed_results["matching_records_dataset1"] >= strict_results["matching_records_dataset1"]
assert relaxed_results["matching_records_dataset2"] >= strict_results["matching_records_dataset2"]
assert relaxed_matches_df["TokenCount"].eq(4).all()

## 6. Inspect one linked record pair

The overlap output links only record IDs. This cell joins one strict-match pair back to the synthetic source datasets so you can see the person attributes that lined up.

In [ ]:
sample_match = strict_matches_df.iloc[0]
hospital_record = hospital_df.loc[hospital_df["RecordId"] == sample_match["HospitalRecordId"]].iloc[0]
pharmacy_record = pharmacy_df.loc[pharmacy_df["RecordId"] == sample_match["PharmacyRecordId"]].iloc[0]

print("Hospital record:")
display(hospital_record.to_frame().T)
print("Pharmacy record:")
display(pharmacy_record.to_frame().T)

assert hospital_record["FirstName"] == pharmacy_record["FirstName"]
assert hospital_record["LastName"] == pharmacy_record["LastName"]
assert hospital_record["BirthDate"] == pharmacy_record["BirthDate"]
assert hospital_record["SocialSecurityNumber"] == pharmacy_record["SocialSecurityNumber"]

## 7. Summary

- The notebook demonstrates the **PySpark bridge** for both token generation and overlap analysis.
- The exchange config is created inside the notebook so the entire bridge workflow is visible.
- Strict matching should produce **40** linked record pairs.
- Relaxed matching should produce **at least** the strict count.

For the separate CLI walkthrough, use `./run_end_to_end.sh` from this directory.